# 12 — ISIC 2019 Preprocessing (FIX 1)

Downloads ISIC 2019 from Kaggle, extracts melanoma (MEL) images only,
runs the same DullRazor + Otsu segmentation + 448×448 resize pipeline
as `00_data_setup`, and saves:

- `MyDrive/melanoma/X_isic2019_mel.npy`  — (N, 448, 448, 3) uint8
- `MyDrive/melanoma/ids_isic2019_mel.npy` — (N,) image_id strings

After running this notebook once, every CNN notebook (04–09) will
automatically merge these images into the training split via
`load_arrays_extended()` in `src/data.py`.

**Val and test splits are untouched (pure HAM10000) — fair benchmark.**

## Expected numbers
- ISIC 2019 total: 25,331 images, 8 classes
- MEL class: ~4,522 images
- After merge: training melanoma 779 + 4522 ≈ 5,301 (vs 6,229 non-mel → near-balanced)

## Runtime
~45 min on Colab A100 (download 10 GB + preprocess 4,500 images).

In [ ]:
# --- Colab setup ---
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/zkoymen/melanoma-detection-ham10000.git"
CANDIDATE_PATHS = [
    Path.cwd(),
    Path("/content/melanoma-detection-ham10000"),
    Path("/content/drive/MyDrive/melanoma-detection-ham10000"),
]

project_root = None
for p in CANDIDATE_PATHS:
    if (p / "src").exists() and (p / "config.py").exists():
        project_root = p
        break

if project_root is None:
    project_root = Path("/content/melanoma-detection-ham10000")
    subprocess.run(["git", "clone", REPO_URL, str(project_root)], check=True)
else:
    subprocess.run(["git", "-C", str(project_root), "pull"], check=True)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except (ImportError, ModuleNotFoundError):
    pass

import config
config.ensure_drive_dirs()
print("Project root:", project_root)
print("Data dir    :", config.DATA_DIR)

In [ ]:
# --- Kaggle auth ---
# Option A: upload kaggle.json manually (run this cell once)
import os
from pathlib import Path

kaggle_json = Path("/root/.kaggle/kaggle.json")
if not kaggle_json.exists():
    try:
        from google.colab import files
        print("Upload your kaggle.json (from kaggle.com -> Account -> API -> Create New Token)")
        uploaded = files.upload()  # user uploads kaggle.json here
        kaggle_json.parent.mkdir(parents=True, exist_ok=True)
        import shutil
        shutil.move(list(uploaded.keys())[0], str(kaggle_json))
        kaggle_json.chmod(0o600)
        print("kaggle.json installed.")
    except Exception as e:
        print(f"Manual upload failed: {e}")
        print("Alternative: set KAGGLE_USERNAME and KAGGLE_KEY as Colab secrets,")
        print("then run: import os; os.environ['KAGGLE_USERNAME']='...'; os.environ['KAGGLE_KEY']='...'")
else:
    print("kaggle.json already present.")

# Option B: use Colab secrets (uncomment if you stored them as secrets)
# from google.colab import userdata
# os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
# os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

In [ ]:
# --- Download ISIC 2019 dataset from Kaggle ---
# Dataset: andrewmvd/isic-2019 (~10 GB unzipped)
# Contains ISIC_2019_Training_Input/ + ISIC_2019_Training_GroundTruth.csv

ISIC_DIR = Path("/content/isic2019")
ISIC_DIR.mkdir(exist_ok=True)

csv_path = ISIC_DIR / "ISIC_2019_Training_GroundTruth.csv"
if not csv_path.exists():
    print("Downloading ISIC 2019 (~10 GB) ...")
    import subprocess
    result = subprocess.run(
        ["kaggle", "datasets", "download",
         "-d", "andrewmvd/isic-2019",
         "-p", str(ISIC_DIR),
         "--unzip"],
        capture_output=True, text=True
    )
    print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr[-1000:])
        raise RuntimeError("Kaggle download failed — check kaggle.json auth.")
else:
    print("ISIC 2019 already downloaded.")

print("Files in ISIC_DIR:")
for f in sorted(ISIC_DIR.iterdir())[:20]:
    print(" ", f.name)

In [ ]:
# --- Parse ground truth, filter MEL class ---
import pandas as pd
import numpy as np

gt = pd.read_csv(csv_path)
print("ISIC 2019 ground truth shape:", gt.shape)
print("Columns:", list(gt.columns))
print("\nClass counts:")
for col in gt.columns[1:]:
    print(f"  {col}: {int(gt[col].sum())}")

# MEL column == 1 means melanoma
mel_ids = gt[gt["MEL"] == 1]["image"].tolist()
print(f"\nMelanoma images: {len(mel_ids)}")

In [ ]:
# --- Find image directory ---
# The dataset may be nested differently depending on Kaggle download version
import os

IMG_DIR = None
for candidate in [
    ISIC_DIR / "ISIC_2019_Training_Input",
    ISIC_DIR / "train",
    ISIC_DIR,
]:
    if candidate.exists() and any(candidate.glob("*.jpg")):
        IMG_DIR = candidate
        break
    # Also check one level deeper
    for sub in candidate.glob("*/"):
        if any(sub.glob("*.jpg")):
            IMG_DIR = sub
            break
    if IMG_DIR:
        break

if IMG_DIR is None:
    # Fall back: find any directory with .jpg files
    for root, dirs, files in os.walk(str(ISIC_DIR)):
        jpgs = [f for f in files if f.endswith(".jpg")]
        if jpgs:
            IMG_DIR = Path(root)
            print(f"Found images in: {IMG_DIR}  ({len(jpgs)} jpgs)")
            break

print(f"Image directory: {IMG_DIR}")
sample_jpgs = list(IMG_DIR.glob("*.jpg"))[:5]
print("Sample files:", [f.name for f in sample_jpgs])

In [ ]:
# --- Preprocess MEL images with same pipeline as HAM10000 ---
import cv2
import numpy as np
import time
from pathlib import Path
from src.preprocessing import preprocess_for_storage

IMG_SIZE = config.IMG_SIZE  # 448

# Verify which MEL images actually exist on disk
mel_paths = []
missing = []
for img_id in mel_ids:
    p = IMG_DIR / f"{img_id}.jpg"
    if p.exists():
        mel_paths.append((img_id, p))
    else:
        missing.append(img_id)

print(f"Found: {len(mel_paths)}  |  Missing: {len(missing)}")
if missing:
    print("First 5 missing:", missing[:5])

# Preprocess
X_isic = np.zeros((len(mel_paths), IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
ids_isic = np.empty(len(mel_paths), dtype=object)
n_fallback = 0
t0 = time.time()

for i, (img_id, img_path) in enumerate(mel_paths):
    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None:
        # Unreadable image — store blank
        ids_isic[i] = img_id
        continue
    try:
        rgb, used_fallback = preprocess_for_storage(img_bgr, size=IMG_SIZE)
        X_isic[i] = rgb
        ids_isic[i] = img_id
        n_fallback += int(used_fallback)
    except Exception as e:
        print(f"  ERROR on {img_id}: {e}")
        ids_isic[i] = img_id

    if (i + 1) % 200 == 0:
        elapsed = time.time() - t0
        rate = (i + 1) / elapsed
        remaining = (len(mel_paths) - i - 1) / rate
        print(f"  {i+1}/{len(mel_paths)}  |  {rate:.1f} img/s  |  ETA {remaining/60:.1f} min")

elapsed = time.time() - t0
print(f"\nDone: {len(mel_paths)} images in {elapsed/60:.1f} min")
print(f"Fallback crops: {n_fallback}/{len(mel_paths)} ({100*n_fallback/len(mel_paths):.1f}%)")
print(f"X_isic shape: {X_isic.shape}  dtype: {X_isic.dtype}")

In [ ]:
# --- Quick visual sanity check (4 random MEL images) ---
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.RandomState(42)
idxs = rng.choice(len(X_isic), size=4, replace=False)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, k in zip(axes, idxs):
    ax.imshow(X_isic[k])
    ax.set_title(ids_isic[k])
    ax.axis("off")
fig.suptitle("ISIC 2019 MEL — preprocessed sample")
plt.tight_layout()
plt.show()

In [ ]:
# --- Save to Drive ---
import numpy as np
from pathlib import Path

OUT_X   = config.DATA_DIR / "X_isic2019_mel.npy"
OUT_IDS = config.DATA_DIR / "ids_isic2019_mel.npy"

print(f"Saving X_isic2019_mel.npy ({X_isic.nbytes / 1e6:.0f} MB) -> {OUT_X}")
np.save(OUT_X, X_isic)
print(f"Saving ids_isic2019_mel.npy -> {OUT_IDS}")
np.save(OUT_IDS, ids_isic)

print("\nDone. Verify files on Drive:")
print(f"  X   : {OUT_X}  ({OUT_X.stat().st_size / 1e6:.0f} MB)")
print(f"  ids : {OUT_IDS}  ({OUT_IDS.stat().st_size / 1e3:.0f} KB)")
print("\nNext step: run notebooks/06_resnet50.ipynb or 07_efficientnet_b3.ipynb")
print("load_arrays_extended() will auto-merge these into training.")

In [ ]:
# --- Verify merge works (dry run of load_arrays_extended) ---
from src.data import load_arrays_extended

X_m, y_m, ids_m, idx_tr, idx_val, idx_test = load_arrays_extended(config.DATA_DIR)

n_mel_train  = int((y_m[idx_tr] == 1).sum())
n_non_train  = int((y_m[idx_tr] == 0).sum())
print(f"Extended training set:")
print(f"  Melanoma     : {n_mel_train}")
print(f"  Non-melanoma : {n_non_train}")
print(f"  Ratio        : 1 : {n_non_train / n_mel_train:.2f}")
print(f"\nVal  images: {len(idx_val)}  (HAM10000 only, unchanged)")
print(f"Test images: {len(idx_test)}  (HAM10000 only, unchanged)")
print(f"\nTotal X shape: {X_m.shape}")